# 유사 상권 기반 프랜차이즈 추천

공개 저장소에서는 합성 Demo Seed를 기본으로 사용합니다. 실제 원천 데이터와 DB credential은 저장소에 포함하지 않습니다.

In [ ]:
import os
from pathlib import Path
import folium
import pandas as pd
import pymysql
from pyproj import Transformer
from sklearn.metrics.pairwise import cosine_similarity

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_MODE = os.getenv('DATA_MODE', 'demo').lower()
DATA_DIR = Path(os.getenv('FRANCHISE_INPUT_DATA_DIR', ROOT / 'data'))
INPUT_FILE = os.getenv('FRANCHISE_INPUT_FILE', '원천동.xlsx')


## DB 연결 및 입력 지역 준비

In [ ]:
conn = pymysql.connect(
    host=os.getenv('FRANCHISE_DB_HOST', '127.0.0.1'),
    port=int(os.getenv('FRANCHISE_DB_PORT', '3306')),
    user=os.getenv('FRANCHISE_DB_USER', 'franchise'),
    password=os.getenv('FRANCHISE_DB_PASSWORD', 'franchise'),
    db=os.getenv('FRANCHISE_DB_NAME', 'franchise_demo'),
    charset='utf8mb4',
)
cursor = conn.cursor()
cursor.execute('SELECT * FROM 상권정보')
rows = cursor.fetchall()
columns = [d[0] for d in cursor.description]
commercial_areas = pd.DataFrame(rows, columns=columns)

if DATA_MODE == 'demo':
    target = commercial_areas.loc[commercial_areas['상권_코드'] == 9000].copy()
    candidates = commercial_areas.loc[commercial_areas['상권_코드'] != 9000].copy()
    if target.empty:
        raise RuntimeError('Demo Target Area(상권_코드=9000)가 없습니다. docker compose down -v 후 다시 초기화하세요.')
else:
    target = pd.DataFrame(pd.read_excel(DATA_DIR / INPUT_FILE), columns=columns)
    candidates = commercial_areas.copy()

target[['상권_코드', '상권_코드_명']]


## 코사인 유사도 기반 유사 상권 탐색

In [ ]:
IDENTIFIER_COLUMNS = ['상권_구분_코드','상권_코드','상권_코드_명','엑스좌표_값','와이좌표_값','자치구_코드','행정동_코드']
feature_target = target.drop(columns=IDENTIFIER_COLUMNS).fillna(0)
feature_areas = candidates.drop(columns=IDENTIFIER_COLUMNS).fillna(0)
scores = cosine_similarity(feature_target.iloc[[0]], feature_areas)[0]
ranked = candidates[['상권_코드','상권_코드_명']].copy()
ranked['cosine_similarity'] = scores
similar_areas = ranked.sort_values('cosine_similarity', ascending=False).head(5).reset_index(drop=True)
similar_areas


## 유사 상권 좌표 조회 및 지도 시각화

In [ ]:
area_codes = similar_areas['상권_코드'].astype(int).tolist()
placeholders = ','.join(['%s'] * len(area_codes))
cursor.execute(
    f'SELECT 상권_코드_명, 엑스좌표_값, 와이좌표_값 FROM 상권정보 WHERE 상권_코드 IN ({placeholders})',
    tuple(area_codes),
)
coordinates = cursor.fetchall()
m = folium.Map(location=[37.56, 126.98], zoom_start=11)
transformer = Transformer.from_crs('EPSG:5181', 'EPSG:4326', always_xy=True)
for name, x, y in coordinates:
    lon, lat = transformer.transform(x, y)
    folium.Marker([lat, lon], tooltip=name).add_to(m)
m


## 유사 상권의 업종 성과 조회

In [ ]:
cursor.execute(
    f'''
    SELECT s.서비스_업종_코드, c.서비스_업종명, SUM(s.매출) AS total_sales, SUM(s.점포_수) AS total_stores
    FROM 점포 s
    JOIN 서비스_업종 c ON c.서비스_업종_코드 = s.서비스_업종_코드
    WHERE s.상권_코드 IN ({placeholders})
    GROUP BY s.서비스_업종_코드, c.서비스_업종명
    ORDER BY total_sales DESC
    ''',
    tuple(area_codes),
)
category_performance = pd.DataFrame(cursor.fetchall(), columns=['서비스_업종_코드','서비스_업종명','total_sales','total_stores'])
category_performance


## 프랜차이즈 후보 조회
가장 높은 매출 업종을 기준으로 프로젝트 당시 휴리스틱 요소를 단순화한 Demo 점수를 계산합니다.

In [ ]:
top_category = category_performance.iloc[0]['서비스_업종_코드']
cursor.execute(
    '''
    SELECT 상호명, 업종코드, 당기순이익, 신규개점, 계약해지, 교육비, 보증금, 기타비용, 단위면적당_인테리어비용,
           (COALESCE(당기순이익, 0) / 1000000.0)
           + COALESCE(신규개점, 0) * 10
           - COALESCE(계약해지, 0) * 10
           - COALESCE(교육비, 0) / 1000000.0
           - COALESCE(보증금, 0) / 1000000.0
           - COALESCE(기타비용, 0) / 1000000.0
           - COALESCE(단위면적당_인테리어비용, 0) / 1000000.0 AS demo_score
    FROM 프랜차이즈
    WHERE 업종코드 = %s
    ORDER BY demo_score DESC
    ''',
    (top_category,),
)
recommendations = pd.DataFrame(
    cursor.fetchall(),
    columns=['상호명','업종코드','당기순이익','신규개점','계약해지','교육비','보증금','기타비용','단위면적당_인테리어비용','demo_score'],
)
recommendations


In [ ]:
cursor.close()
conn.close()
